<a href="https://colab.research.google.com/github/aparna-2001/GATE_PPI/blob/main/mapping_ID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd


In [ ]:
os.makedirs('data/raw', exist_ok=True)

# STRING provides an official alias file that maps
# Ensembl protein IDs to UniProt accessions and many other ID types
!wget -O data/raw/9606.protein.aliases.v12.0.txt.gz \
  "https://stringdb-downloads.org/download/protein.aliases.v12.0/9606.protein.aliases.v12.0.txt.gz"

--2026-06-17 09:44:05--  https://stringdb-downloads.org/download/protein.aliases.v12.0/9606.protein.aliases.v12.0.txt.gz
Resolving stringdb-downloads.org (stringdb-downloads.org)... 49.12.123.75
Connecting to stringdb-downloads.org (stringdb-downloads.org)|49.12.123.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 19777800 (19M) [application/octet-stream]
Saving to: ‘data/raw/9606.protein.aliases.v12.0.txt.gz’

data/raw/9606.prote 100%[===================>]  18.86M  9.93MB/s    in 1.9s    

2026-06-17 09:44:08 (9.93 MB/s) - ‘data/raw/9606.protein.aliases.v12.0.txt.gz’ saved [19777800/19777800]



In [ ]:
aliases = pd.read_csv(
    'data/raw/9606.protein.aliases.v12.0.txt.gz',
    sep='\t',
    compression='gzip'
)

print(f"Shape: {aliases.shape}")
print(f"\nColumns: {aliases.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(aliases.head())
print(f"\nUnique sources:")
print(aliases['source'].value_counts().head(20))

Shape: (3889207, 3)

Columns: ['#string_protein_id', 'alias', 'source']

First 5 rows:
     #string_protein_id alias                  source
0  9606.ENSP00000000233  2B6H             Ensembl_PDB
1  9606.ENSP00000000233  2B6H          UniProt_DR_PDB
2  9606.ENSP00000000233   381  Ensembl_HGNC_entrez_id
3  9606.ENSP00000000233   381             KEGG_GENEID
4  9606.ENSP00000000233   381       KEGG_KEGGID_SHORT

Unique sources:
source
Ensembl_EMBL                     487265
Ensembl_protein_id               455461
Ensembl_RefSeq_short             225974
Ensembl_RefSeq                   225974
Ensembl_transcript               152417
Ensembl_HGNC_trans_name          151338
Ensembl_UCSC                     148513
Ensembl_archive_transcript       144556
Ensembl_archive_translation      125205
Ensembl_Reactome_gene            110630
UniProt_DR_RefSeq                109660
Ensembl_Reactome                 108642
Ensembl_translation               99376
Ensembl_UniProt                   96363
UniPr

In [ ]:
# Filter to UniProt_AC source only
uniprot_map = aliases[aliases['source'] == 'UniProt_AC'][
    ['#string_protein_id', 'alias']
].copy()

# Rename columns for clarity
uniprot_map.columns = ['string_id', 'uniprot_id']

print(f"Mapping rows: {len(uniprot_map):,}")
print(f"\nFirst 5 rows:")
print(uniprot_map.head())

# Check if any STRING ID maps to multiple UniProt IDs
duplicated_string_ids = uniprot_map['string_id'].duplicated().sum()
print(f"\nSTRING IDs mapping to multiple UniProt IDs: {duplicated_string_ids:,}")

Mapping rows: 88,155

First 5 rows:
                string_id uniprot_id
83   9606.ENSP00000000233     P26437
86   9606.ENSP00000000233     P84085
123  9606.ENSP00000000412     A8K528
148  9606.ENSP00000000412     D3DUV5
264  9606.ENSP00000000412     P20645

STRING IDs mapping to multiple UniProt IDs: 68,756


In [ ]:
# Flag Swiss-Prot entries — accessions starting with P, Q, or O
uniprot_map['is_swissprot'] = uniprot_map['uniprot_id'].str[0].isin(['P', 'Q', 'O'])

print(f"Total mappings:           {len(uniprot_map):,}")
print(f"Swiss-Prot (reviewed):    {uniprot_map['is_swissprot'].sum():,}")
print(f"TrEMBL (unreviewed):      {(~uniprot_map['is_swissprot']).sum():,}")

# Keep Swiss-Prot entries first, fall back to TrEMBL if no Swiss-Prot exists
# Sort so Swiss-Prot entries come first for each string_id
uniprot_map_sorted = uniprot_map.sort_values(
    ['string_id', 'is_swissprot'],
    ascending=[True, False]   # string_id ascending, is_swissprot descending (True first)
)

# Keep first occurrence per string_id — which is now the Swiss-Prot entry if one exists
uniprot_map_final = uniprot_map_sorted.drop_duplicates(
    subset='string_id',
    keep='first'
).reset_index(drop=True)

print(f"\nAfter deduplication:")
print(f"Unique STRING IDs mapped: {len(uniprot_map_final):,}")
print(f"Swiss-Prot entries kept:  {uniprot_map_final['is_swissprot'].sum():,}")
print(f"TrEMBL fallbacks kept:    {(~uniprot_map_final['is_swissprot']).sum():,}")
print(f"\nFirst 5 rows:")
print(uniprot_map_final[['string_id', 'uniprot_id', 'is_swissprot']].head())

Total mappings:           88,155
Swiss-Prot (reviewed):    66,316
TrEMBL (unreviewed):      21,839

After deduplication:
Unique STRING IDs mapped: 19,399
Swiss-Prot entries kept:  18,302
TrEMBL fallbacks kept:    1,097

First 5 rows:
              string_id uniprot_id  is_swissprot
0  9606.ENSP00000000233     P26437          True
1  9606.ENSP00000000412     P20645          True
2  9606.ENSP00000001008     Q02790          True
3  9606.ENSP00000001146     Q32MC0          True
4  9606.ENSP00000002125     Q7L592          True


In [ ]:
pairs_combined = pd.read_parquet('/content/pairs_combined.parquet')

# Split pairs_combined into two groups by ID type
string_pairs   = pairs_combined[pairs_combined['source'] != 'Negatome_combined_stringent'].copy()
negatome_pairs = pairs_combined[pairs_combined['source'] == 'Negatome_combined_stringent'].copy()

print(f"STRING + random pairs (need mapping): {len(string_pairs):,}")
print(f"Negatome pairs (already UniProt):     {len(negatome_pairs):,}")

# Build a simple dictionary for fast lookup
mapping_dict = dict(zip(uniprot_map_final['string_id'], uniprot_map_final['uniprot_id']))

# Map protein1 and protein2
string_pairs['protein1'] = string_pairs['protein1'].map(mapping_dict)
string_pairs['protein2'] = string_pairs['protein2'].map(mapping_dict)

# Check how many failed to map (NaN means no UniProt ID found)
failed_p1 = string_pairs['protein1'].isnull().sum()
failed_p2 = string_pairs['protein2'].isnull().sum()
print(f"\nFailed to map protein1: {failed_p1:,}")
print(f"Failed to map protein2: {failed_p2:,}")

STRING + random pairs (need mapping): 36,890
Negatome pairs (already UniProt):     0

Failed to map protein1: 1,525
Failed to map protein2: 2,350


In [ ]:
# Find all rows where either protein failed to map
failed_mask = string_pairs['protein1'].isnull() | string_pairs['protein2'].isnull()

print(f"Pairs where protein1 failed:          {string_pairs['protein1'].isnull().sum():,}")
print(f"Pairs where protein2 failed:          {string_pairs['protein2'].isnull().sum():,}")
print(f"Pairs where EITHER failed (to drop):  {failed_mask.sum():,}")
print(f"Pairs where BOTH failed:              {(string_pairs['protein1'].isnull() & string_pairs['protein2'].isnull()).sum():,}")

# Check label distribution of failed pairs
print(f"\nLabel distribution of failed pairs:")
print(string_pairs[failed_mask]['label'].value_counts())
print(f"\nSource distribution of failed pairs:")
print(string_pairs[failed_mask]['source'].value_counts())

Pairs where protein1 failed:          1,525
Pairs where protein2 failed:          2,350
Pairs where EITHER failed (to drop):  2,466
Pairs where BOTH failed:              1,409

Label distribution of failed pairs:
label
0    2074
1     392
Name: count, dtype: int64

Source distribution of failed pairs:
source
Negatome_combined_stringent_human    1406
random_negative                       668
STRING_experimental                   392
Name: count, dtype: int64


In [ ]:
# Drop failed pairs from string_pairs
string_pairs_clean = string_pairs[~failed_mask].copy()
print(f"String pairs after dropping failures: {len(string_pairs_clean):,}")

# Recombine with negatome pairs (which never needed mapping)
pairs_mapped = pd.concat(
    [string_pairs_clean, negatome_pairs],
    ignore_index=True
)

print(f"\nTotal pairs after mapping: {len(pairs_mapped):,}")
print(f"\nLabel distribution:")
print(pairs_mapped['label'].value_counts())
print(f"\nRatio: {(pairs_mapped['label']==0).sum() / (pairs_mapped['label']==1).sum():.4f}")
print(f"\nFirst 5 rows:")
print(pairs_mapped.head())
print(f"\nSample of negatome rows (already UniProt):")
print(pairs_mapped[pairs_mapped['source']=='Negatome_combined_stringent'].head(3))

String pairs after dropping failures: 34,424

Total pairs after mapping: 34,424

Label distribution:
label
0    24411
1    10013
Name: count, dtype: int64

Ratio: 2.4379

First 5 rows:
  protein1 protein2  label               source
0   P26437   P18085      1  STRING_experimental
1   P26437   P10947      1  STRING_experimental
2   P20645   Q15017      1  STRING_experimental
3   Q02790   Q5VVC3      1  STRING_experimental
4   Q02790   P42345      1  STRING_experimental

Sample of negatome rows (already UniProt):
Empty DataFrame
Columns: [protein1, protein2, label, source]
Index: []


In [ ]:
import json
from datetime import datetime

# Final checks
print(f"Any nulls: {pairs_mapped.isnull().sum().sum()}")
print(f"Any duplicates: {pairs_mapped.duplicated(subset=['protein1','protein2']).sum()}")
print(f"Any self-pairs: {(pairs_mapped['protein1']==pairs_mapped['protein2']).sum()}")

# Verify all IDs look like UniProt accessions
# UniProt accessions are 6 or 10 characters, start with a letter
import re
uniprot_pattern = re.compile(r'^[A-Z][0-9][A-Z0-9]{3}[0-9]')
invalid_p1 = pairs_mapped['protein1'].apply(
    lambda x: not bool(uniprot_pattern.match(str(x)))
).sum()
invalid_p2 = pairs_mapped['protein2'].apply(
    lambda x: not bool(uniprot_pattern.match(str(x)))
).sum()
print(f"Invalid UniProt format in protein1: {invalid_p1}")
print(f"Invalid UniProt format in protein2: {invalid_p2}")

Any nulls: 0
Any duplicates: 69
Any self-pairs: 0
Invalid UniProt format in protein1: 0
Invalid UniProt format in protein2: 0


In [ ]:
# Step 1 — Remove self-pairs
before = len(pairs_mapped)
pairs_mapped = pairs_mapped[
    pairs_mapped['protein1'] != pairs_mapped['protein2']
].copy()
print(f"Removed self-pairs: {before - len(pairs_mapped):,}")

# Step 2 — Remove invalid UniProt format
import re
uniprot_pattern = re.compile(r'^[A-Z][0-9][A-Z0-9]{3}[0-9]')

valid_mask = (
    pairs_mapped['protein1'].apply(lambda x: bool(uniprot_pattern.match(str(x)))) &
    pairs_mapped['protein2'].apply(lambda x: bool(uniprot_pattern.match(str(x))))
)
before = len(pairs_mapped)
pairs_mapped = pairs_mapped[valid_mask].copy()
print(f"Removed invalid UniProt IDs: {before - len(pairs_mapped):,}")

# Step 3 — Remove duplicates (keep first)
before = len(pairs_mapped)
pairs_mapped = pairs_mapped.drop_duplicates(
    subset=['protein1', 'protein2'],
    keep='first'
).reset_index(drop=True)
print(f"Removed duplicates: {before - len(pairs_mapped):,}")

# Final counts
print(f"\nTotal pairs remaining:  {len(pairs_mapped):,}")
print(f"Positives:              {(pairs_mapped['label']==1).sum():,}")
print(f"Negatives:              {(pairs_mapped['label']==0).sum():,}")
print(f"Ratio:                  {(pairs_mapped['label']==0).sum() / (pairs_mapped['label']==1).sum():.4f}")

Removed self-pairs: 0
Removed invalid UniProt IDs: 0
Removed duplicates: 69

Total pairs remaining:  34,355
Positives:              9,944
Negatives:              24,411
Ratio:                  2.4548


In [ ]:
# Final verification — all five checks must be 0
print(f"Nulls:        {pairs_mapped.isnull().sum().sum()}")
print(f"Duplicates:   {pairs_mapped.duplicated(subset=['protein1','protein2']).sum()}")
print(f"Self-pairs:   {(pairs_mapped['protein1']==pairs_mapped['protein2']).sum()}")
invalid_p1 = pairs_mapped['protein1'].apply(
    lambda x: not bool(uniprot_pattern.match(str(x)))
).sum()
invalid_p2 = pairs_mapped['protein2'].apply(
    lambda x: not bool(uniprot_pattern.match(str(x)))
).sum()
print(f"Invalid p1:   {invalid_p1}")
print(f"Invalid p2:   {invalid_p2}")

Nulls:        0
Duplicates:   0
Self-pairs:   0
Invalid p1:   0
Invalid p2:   0


In [ ]:
import json
from datetime import datetime

# Use the ACTUAL values from this run, not hardcoded old ones
metadata = {
    "date_created":             datetime.today().strftime('%Y-%m-%d'),
    "input_pairs":               36890,   # actual pairs_combined count after Negatome correction
    "unmappable_dropped":        None,    # fill in from your earlier failed_mask.sum() output in this run
    "self_pairs_dropped":        0,
    "invalid_uniprot_dropped":   0,
    "duplicates_dropped":        69,
    "total_pairs":                len(pairs_mapped),
    "positives":                  int((pairs_mapped['label']==1).sum()),
    "negatives":                  int((pairs_mapped['label']==0).sum()),
    "ratio":                      round((pairs_mapped['label']==0).sum() /
                                        (pairs_mapped['label']==1).sum(), 4),
    "id_space":                   "UniProt accession (all proteins)",
    "swiss_prot_mappings":        18302,
    "trembl_fallbacks":           1097,
    "correction_note": "This mapping was rerun after Negatome was corrected to human-only proteins. Input pairs_combined (36,890) reflects the corrected merge, not the original contaminated version (41,292).",
    "checks_passed": {
        "nulls":        0,
        "duplicates":   0,
        "self_pairs":   0,
        "invalid_ids":  0
    }
}

with open('data/raw/pairs_mapped_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Corrected metadata saved")
print(json.dumps(metadata, indent=2))

Corrected metadata saved
{
  "date_created": "2026-06-17",
  "input_pairs": 36890,
  "unmappable_dropped": null,
  "self_pairs_dropped": 0,
  "invalid_uniprot_dropped": 0,
  "duplicates_dropped": 69,
  "total_pairs": 34355,
  "positives": 9944,
  "negatives": 24411,
  "ratio": 2.4548,
  "id_space": "UniProt accession (all proteins)",
  "swiss_prot_mappings": 18302,
  "trembl_fallbacks": 1097,
  "correction_note": "This mapping was rerun after Negatome was corrected to human-only proteins. Input pairs_combined (36,890) reflects the corrected merge, not the original contaminated version (41,292).",
  "checks_passed": {
    "nulls": 0,
    "duplicates": 0,
    "self_pairs": 0,
    "invalid_ids": 0
  }
}


In [ ]:
unique_proteins = set(pairs_mapped['protein1']) | set(pairs_mapped['protein2'])
print(f"Unique proteins needing structures: {len(unique_proteins):,}")

Unique proteins needing structures: 3,598
